In [11]:


import pandas as pd

userbase = pd.read_csv("../data/recommendations.csv")
print(userbase.shape)  # (125855, 40) 이 나오면 정상
print(userbase.head())

(41154794, 8)
    app_id  helpful  funny        date  is_recommended  hours  user_id  \
0   975370        0      0  2022-12-12            True   36.3    51580   
1   304390        4      0  2017-02-17           False   11.5     2586   
2  1085660        2      0  2019-11-17            True  336.5   253880   
3   703080        0      0  2022-09-23            True   27.4   259432   
4   526870        0      0  2021-01-10            True    7.9    23869   

   review_id  
0          0  
1          1  
2          2  
3          3  
4          4  


In [12]:
user_review_counts = userbase.groupby('user_id').size().reset_index(name='review_count')
user_review_counts = user_review_counts.sort_values('review_count', ascending=False)
print(user_review_counts.head())

           user_id  review_count
11310572  11764552          6045
4801534    5112758          4152
11203911  11656130          3840
5348684    5669734          3479
11103125  11553593          3392


In [14]:
lower_bound = 10
upper_bound = 78  # 99.9 percentile 근사값 (봇/이상치 배제)

eligible_users = user_review_counts[
    (user_review_counts['review_count'] >= lower_bound) &
    (user_review_counts['review_count'] <= upper_bound)
]
print(f"평가 대상 유저 수: {len(eligible_users):,}")

평가 대상 유저 수: 666,781


In [15]:
import pandas as pd
from scipy.sparse import csr_matrix

# 고유 user_id, app_id에 0부터 순번 매기기
unique_users = userbase['user_id'].unique()
unique_games = userbase['app_id'].unique()

user_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
game_to_idx = {aid: idx for idx, aid in enumerate(unique_games)}

In [16]:
userbase['score'] = userbase['is_recommended'].map({True: 1, False: -1})

In [17]:
row_idx = userbase['user_id'].map(user_to_idx).values
col_idx = userbase['app_id'].map(game_to_idx).values
values = userbase['score'].values

interaction_matrix = csr_matrix(
    (values, (row_idx, col_idx)),
    shape=(len(unique_users), len(unique_games))
)

In [18]:
print(interaction_matrix.shape)   # (유저 수, 게임 수)
print(interaction_matrix.nnz)     # 실제 저장된 non-zero 값 개수

(13781059, 37610)
41154773


In [29]:
import sys
import os
from sklearn.metrics.pairwise import cosine_similarity

sys.path.append(os.path.abspath(".."))
from evaluation import build_user_review_groups, stratified_sample_users

eligible_users = build_user_review_groups(userbase)  # lower=10, upper=78 그대로
sampled_users = stratified_sample_users(eligible_users, sample_per_group=100, random_state=42)
sample_indices = [user_to_idx[uid] for uid in sampled_users['user_id']]

sample_matrix = interaction_matrix[sample_indices]

user_sim = cosine_similarity(sample_matrix, interaction_matrix, dense_output=False)

# 자기 자신 유사도 제거
for i, orig_idx in enumerate(sample_indices):
    user_sim[i, orig_idx] = 0

평가 대상 유저 수: 666,781
count    666781.000000
mean         18.842848
std          11.465816
min          10.000000
25%          11.000000
50%          15.000000
75%          21.000000
max          78.000000
Name: n_games, dtype: float64
review_group
10-15개    366006
16-25개    181674
26-45개     89262
46-78개     29839
Name: count, dtype: int64
review_group
10-15개    100
16-25개    100
26-45개    100
46-78개    100
Name: count, dtype: int64


In [30]:
print(user_sim.shape)  # (샘플 유저 수, 전체 유저 수)
print(user_sim.nnz)    # 실제 저장된 non-zero 값 개수
print(user_sim)  # 일부 샘플 유저의 유사도 확인

(400, 13781059)
379541639
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 379541639 stored elements and shape (400, 13781059)>
  Coords	Values
  (0, 13451764)	0.2886751345948129
  (0, 13446923)	0.2886751345948129
  (0, 13446636)	0.2886751345948129
  (0, 13446100)	0.2041241452319315
  (0, 13443326)	0.2886751345948129
  (0, 13440069)	0.2886751345948129
  (0, 13439889)	0.2041241452319315
  (0, 13435328)	-0.2886751345948129
  (0, 13430942)	0.2886751345948129
  (0, 13430891)	0.2886751345948129
  (0, 13429177)	0.1666666666666667
  (0, 13429153)	0.2886751345948129
  (0, 13428565)	0.2886751345948129
  (0, 13426708)	0.2886751345948129
  (0, 13426422)	0.1666666666666667
  (0, 13424341)	0.2041241452319315
  (0, 13424072)	0.09622504486493763
  (0, 13421814)	0.2886751345948129
  (0, 13420358)	0.2886751345948129
  (0, 13414701)	0.2041241452319315
  (0, 13412854)	0.2886751345948129
  (0, 13410672)	0.2886751345948129
  (0, 13409397)	0.1091089451179962
  (0, 13408842)	0.2886751345948129
 

In [31]:
import numpy as np

K = 30  # 이웃 수, 나중에 튜닝 가능

def get_topk_neighbors(user_sim, k=K):
    """
    user_sim: (400, n_users) sparse matrix, 자기 자신은 이미 0 처리됨
    반환: 각 row(샘플 유저)마다 (이웃 인덱스, 유사도) 리스트
    """
    topk_results = []
    
    for i in range(user_sim.shape[0]):
        row = user_sim.getrow(i)          # sparse row 하나만 꺼냄
        row_indices = row.indices          # 0이 아닌 col 인덱스들
        row_data = row.data                # 그 인덱스들의 유사도 값
        
        if len(row_data) == 0:
            topk_results.append(([], []))
            continue
        
        # 상위 k개만 뽑기 (전체 정렬 대신 argpartition으로 효율적으로)
        if len(row_data) > k:
            top_k_idx = np.argpartition(row_data, -k)[-k:]
        else:
            top_k_idx = np.arange(len(row_data))
        
        # 실제 유사도 기준 내림차순 정렬
        top_k_idx = top_k_idx[np.argsort(-row_data[top_k_idx])]
        
        neighbor_indices = row_indices[top_k_idx]
        neighbor_sims = row_data[top_k_idx]
        
        topk_results.append((neighbor_indices, neighbor_sims))
    
    return topk_results

topk_neighbors = get_topk_neighbors(user_sim, k=K)

In [32]:
print(topk_neighbors[0])  # 첫 번째 샘플 유저의 이웃 인덱스와 유사도 확인

(array([ 8825777,  7170489,  1779824,  6252190,  1254212,  5949411,
       10791410,  4678910,  6316071,  4656514,   148535,  6888677,
        5056647,  1918738,  1226869,  1229104,  7288249, 10436306,
         828317,  3930667,  2367686,  3182708,   362681,  4992045,
       10575806,  4923933, 10868095, 13314585,  4981682,  4984495],
      dtype=int32), array([0.5       , 0.5       , 0.5       , 0.4330127 , 0.4330127 ,
       0.4330127 , 0.40824829, 0.40824829, 0.40824829, 0.40824829,
       0.40824829, 0.40824829, 0.40824829, 0.40824829, 0.40824829,
       0.40824829, 0.40824829, 0.40824829, 0.40824829, 0.40824829,
       0.40824829, 0.40824829, 0.40824829, 0.40824829, 0.40824829,
       0.40824829, 0.40824829, 0.40824829, 0.40824829, 0.40824829]))


In [33]:
def predict_scores_for_user(user_idx, neighbor_indices, neighbor_sims, interaction_matrix):
    """
    특정 샘플 유저 하나에 대해, 모든 게임에 대한 예측 점수를 계산
    """
    if len(neighbor_indices) == 0:
        return np.zeros(interaction_matrix.shape[1])
    
    # 이웃들의 interaction row만 추출 (K x n_games)
    neighbor_matrix = interaction_matrix[neighbor_indices]
    
    # 가중합: sim × neighbor의 점수, 게임(column)별로 합산
    weighted_sum = neighbor_sims @ neighbor_matrix  # (n_games,) 벡터로 나옴
    
    # 정규화: 유사도 절댓값 합으로 나눔
    sim_sum = np.abs(neighbor_sims).sum()
    predicted = weighted_sum / sim_sum if sim_sum > 0 else weighted_sum
    
    return predicted  # 이 유저에 대한 게임별 예측 점수 (n_games,)

In [34]:
def recommend_topn(user_idx, predicted_scores, interaction_matrix, n=10):
    already_played = interaction_matrix[user_idx].indices  # 이미 interaction 있는 게임 idx
    
    predicted_scores = predicted_scores.copy()
    predicted_scores[already_played] = -np.inf  # 이미 한 건 추천 후보에서 제외
    
    top_n_idx = np.argpartition(predicted_scores, -n)[-n:]
    top_n_idx = top_n_idx[np.argsort(-predicted_scores[top_n_idx])]
    
    return top_n_idx  # 추천 게임의 idx 리스트

In [35]:
# 0번째 샘플 유저 기준 테스트
sample_pos = 0
user_idx = sample_indices[sample_pos]  # 원래 interaction_matrix에서의 인덱스
neighbor_idx, neighbor_sim = topk_neighbors[sample_pos]

# 4단계: 예측 점수 계산
predicted_scores = predict_scores_for_user(
    user_idx, neighbor_idx, neighbor_sim, interaction_matrix
)

# 5단계: 이미 한 게임 제외하고 top-N 추천
top_n_idx = recommend_topn(user_idx, predicted_scores, interaction_matrix, n=10)

# 결과를 실제 게임 이름으로 확인
idx_to_game = {v: k for k, v in game_to_idx.items()}
recommended_app_ids = [idx_to_game[idx] for idx in top_n_idx]

print(f"유저 {sampled_users['user_id'].iloc[sample_pos]}에게 추천:")
for app_id, idx in zip(recommended_app_ids, top_n_idx):
    print(f"  app_id={app_id}, predicted_score={predicted_scores[idx]:.4f}")

유저 12185468에게 추천:
  app_id=294810, predicted_score=0.0344
  app_id=239140, predicted_score=0.0344
  app_id=1360660, predicted_score=0.0000
  app_id=534380, predicted_score=0.0000
  app_id=951670, predicted_score=0.0000
  app_id=1550870, predicted_score=0.0000
  app_id=635260, predicted_score=0.0000
  app_id=15380, predicted_score=0.0000
  app_id=392160, predicted_score=0.0000
  app_id=570, predicted_score=0.0000
